# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadaFouad461/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

My unit of analysis is one content item for one client on one report date.

I will use February 2026 as the feature window and March 2026 as the observed outcome window. The goal is to rank content items by their observed future performance signal.

In [23]:
%pip -q install duckdb huggingface_hub

import getpass
import duckdb
from huggingface_hub import login, hf_hub_download, list_repo_files

#  Enter HF Token manually
HF_TOKEN = getpass.getpass("Enter Hugging Face Token: ")

# Authenticate session
login(token=HF_TOKEN)

REPO_ID = "FlyRank/internship-warehouse"

print("Fetching file repository list...")
all_files = list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)

feb_files = [f for f in all_files if "month=2026-02" in f and f.endswith(".parquet")]
mar_files = [f for f in all_files if "month=2026-03" in f and f.endswith(".parquet")]

print("Downloading parquet files locally...")
local_feb_paths = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN) for f in feb_files]
local_mar_paths = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN) for f in mar_files]

#  Connect DuckDB to local files
con = duckdb.connect()

FACT_FEB = f"read_parquet({local_feb_paths})"
FACT_MAR = f"read_parquet({local_mar_paths})"

print(" Files downloaded and connection ready!")

test = con.sql(f"SELECT COUNT(*) AS total_rows FROM {FACT_FEB}").df()
display(test)

Enter Hugging Face Token: ··········
Fetching file repository list...
 Files downloaded and connection ready!


,total_rows
0,7355108


In [18]:
print("Feature window: February 2026")
print("Outcome window: March 2026")

print("February rows:", con.sql(
    f"SELECT COUNT(*) AS rows FROM {FACT_FEB}"
).df())

print("March rows:", con.sql(
    f"SELECT COUNT(*) AS rows FROM {FACT_MAR}"
).df())

Feature window: February 2026
Outcome window: March 2026
February rows:       rows
0  7355108
March rows:       rows
0  9841378


## 2. Fields: feature / label / context / excluded

### Features
I will use February signals that are available at the decision moment:
impressions_90d, clicks_90d, sessions_90d, avg_position, and search_volume.

### Label / ranking proxy
The ranking proxy is March trend_pct, used as an observed future performance signal. It is measured after the February decision point.

### Context
content_type, main_intent, age_tier, freshness_tier, impression_tier, position_tier, and trend_direction are context fields that help describe the content.

### Excluded
client_hash_id and content_hash_id are excluded from model features because they are pseudonymous identifiers. March outcome fields are also excluded from February features to avoid future information leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
#Query 1 — Grain
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS n
FROM {FACT_FEB}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate client-content-date rows:")
display(grain_check)

if grain_check.empty:
    print("Grain check passed: one row per client × content × day.")
else:
    print("Duplicate rows found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate client-content-date rows:


,client_hash_id,content_hash_id,report_date,n


Grain check passed: one row per client × content × day.


In [20]:
#Query 2 — Row count + date span
window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents
FROM {FACT_FEB}
""").df()

display(window_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date,clients,contents
0,7355108,2026-02-01,2026-02-28,54,321546


In [21]:
#Query 3 — Availability
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS unavailable_rows
FROM {FACT_FEB}
""").df()

display(availability_check)

,total_rows,available_rows,unavailable_rows
0,7355108,2621783,4733325


## 4. Data limits


This slice is limited by the available reporting history and the overlap between rolling performance windows. February is used for features and March is treated as the observed outcome window, so the result is directional rather than causal. The data can support ranking and decision-support, but it cannot explain Google's algorithm or prove that a feature caused a ranking change.

In [22]:
print("Limitation recorded: rolling windows overlap, and the result is directional decision-support rather than causal evidence.")

Limitation recorded: rolling windows overlap, and the result is directional decision-support rather than causal evidence.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.